# Fine-tune DistilBERT on WELFake (fake vs real classifier)

**Why this notebook exists:** the TF-IDF + LogReg baseline scores 97% on WELFake, but its top features are source tells — `reuters`, `breitbart`, `getty images`, `featured image`. The model is fingerprinting publishers, not detecting fake news. This notebook scrubs source tells from the text first, then fine-tunes DistilBERT so the model has to rely on language patterns. Expect honest accuracy ~80–92%.

**Hardware:** designed for a single GPU. With 96GB VRAM you can swap `MODEL_NAME` to `roberta-large` or `microsoft/deberta-v3-large` and bump `MAX_LEN` to 512 / `BATCH_SIZE` to 64–128 for stronger results.

## 1. Install dependencies

Uncomment and run once if these aren't already installed in the environment.

In [ ]:
# %pip install "transformers>=4.46" "torch>=2.2" "datasets>=2.18" scikit-learn pandas accelerate

## 2. Imports

In [1]:
from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/jupyterhub/jupyter_env/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/opt/jupyterhub/jupyter_env/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/opt/jupyterhub/jupyter_env/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 758, in start
    self.io_loop.start()
  Fi

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/jupyterhub/jupyter_env/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/opt/jupyterhub/jupyter_env/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/opt/jupyterhub/jupyter_env/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 758, in start
    self.io_loop.start()
  Fi

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



CUDA available: True
Device: NVIDIA RTX A6000
VRAM (GB): 50.9


## 3. Config

Tweak these for your hardware. Defaults match a small GPU; comments show the upgrades for a 96GB card.

In [2]:
CSV_PATH    = "WELFake_Dataset.csv"
OUT_DIR     = Path("distilbert_welfake")
MODEL_NAME  = "distilbert-base-uncased"   # try "roberta-large" or "microsoft/deberta-v3-large" on 96GB
EPOCHS      = 3
BATCH_SIZE  = 32                           # 64–128 fine on 96GB
MAX_LEN     = 256                          # 512 captures full article body on 96GB
LR          = 2e-5
SEED        = 42

OUT_DIR.mkdir(parents=True, exist_ok=True)

## 4. Source-fingerprint scrubbing

These regexes strip the leakage signals — publisher names, datelines, photo credits, URLs, hashtags, twitter handles, and 4-digit years (since 2016/2017 are heavily skewed in WELFake).

In [3]:
SOURCE_PATTERNS = [
    r"\b(reuters|breitbart|bloomberg|associated press|\bap\b|afp|cnn|fox news|"
    r"new york times|nyt|washington post|wapo|the guardian|huffpost|"
    r"infowars|the onion|daily caller|daily wire|buzzfeed|vox|politico|axios|"
    r"the hill|npr|bbc|al jazeera|rt\.com|sputnik|getty|getty images)\b",
    r"\([A-Z]{2,}\)\s*[-—–]",                                    # (REUTERS) -
    r"\b[A-Z][A-Z]+,?\s+[A-Z][a-z]+\.?\s*\d*\s*\(reuters\)",       # WASHINGTON (Reuters)
    r"\bfeatured image\b.*?(?=\.|$)",                              # 'Featured image via ...'
    r"\bphoto[s]?\s+(by|via|credit|courtesy)[^\.]*",
    r"\bimage[s]?\s+(by|via|credit|courtesy)[^\.]*",
    r"\bvia\s+[A-Z][A-Za-z]+(\s+[A-Z][A-Za-z]+){0,2}",
    r"https?://\S+",
    r"www\.\S+",
    r"@[A-Za-z0-9_]{2,}",
    r"#[A-Za-z0-9_]{2,}",
    r"\b\d{4}\b",                                                  # years
]
SOURCE_RE = re.compile("|".join(SOURCE_PATTERNS), flags=re.IGNORECASE)
WHITESPACE_RE = re.compile(r"\s+")


def clean_text(s):
    if not isinstance(s, str):
        return ""
    s = SOURCE_RE.sub(" ", s)
    s = WHITESPACE_RE.sub(" ", s).strip()
    return s


# Quick sanity check — should remove the source tells
sample = "WASHINGTON (Reuters) - President said on Tuesday that... Featured image via Getty Images. https://example.com/foo @realDonaldTrump #news 2017"
print("BEFORE:", sample)
print("AFTER :", clean_text(sample))

BEFORE: WASHINGTON (Reuters) - President said on Tuesday that... Featured image via Getty Images. https://example.com/foo @realDonaldTrump #news 2017
AFTER : WASHINGTON President said on Tuesday that... .


## 5. Load and clean WELFake

In [4]:
df = pd.read_csv(CSV_PATH)
df = df.dropna(subset=["text", "label"]).copy()
df["title"] = df["title"].fillna("")
df["combined"] = (df["title"] + " " + df["text"]).map(clean_text)
df = df[df["combined"].str.len() > 50].reset_index(drop=True)
df["label"] = df["label"].astype(int)
df = df[["combined", "label"]]

print(f"Rows after cleaning: {len(df):,}")
print(df["label"].value_counts().rename({0: "real", 1: "fake"}))

Rows after cleaning: 71,932
label
fake    36905
real    35027
Name: count, dtype: int64


## 6. Train / test split (stratified 80/20)

In [5]:
train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df["label"], random_state=SEED
)
print(f"Train: {len(train_df):,}   Test: {len(test_df):,}")

Train: 57,545   Test: 14,387


## 7. Tokenizer & model

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "real", 1: "fake"},
    label2id={"real": 0, "fake": 1},
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 8. Tokenize

In [7]:
def tokenize(batch):
    return tokenizer(batch["combined"], truncation=True, max_length=MAX_LEN)

train_ds = Dataset.from_pandas(train_df, preserve_index=False).map(
    tokenize, batched=True, remove_columns=["combined"]
)
test_ds = Dataset.from_pandas(test_df, preserve_index=False).map(
    tokenize, batched=True, remove_columns=["combined"]
)

Map:   0%|          | 0/57545 [00:00<?, ? examples/s]

Map:   0%|          | 0/14387 [00:00<?, ? examples/s]

## 9. Training arguments + Trainer

In [8]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }


training_args = TrainingArguments(
    output_dir=str(OUT_DIR / "checkpoints"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


## 10. Train

In [9]:
trainer.train()

/opt/jupyterhub/jupyter_env/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.076295,0.069906,0.990130,0.990123
2,0.047517,0.057247,0.992841,0.992837
3,0.010566,0.063116,0.993744,0.993741


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/opt/jupyterhub/jupyter_env/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/opt/jupyterhub/jupyter_env/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=2700, training_loss=0.10858294202221765, metrics={'train_runtime': 532.2929, 'train_samples_per_second': 324.323, 'train_steps_per_second': 5.072, 'total_flos': 1.143425468348928e+16, 'train_loss': 0.10858294202221765, 'epoch': 3.0})

## 11. Evaluate

In [13]:
# Drop the NotebookProgressCallback — its training_tracker is reset after
# train() finishes and crashes on standalone evaluate()/predict() calls.
from transformers.utils.notebook import NotebookProgressCallback
trainer.remove_callback(NotebookProgressCallback)

preds = trainer.predict(test_ds)
y_pred = np.argmax(preds.predictions, axis=-1)
y_true = preds.label_ids

print("Eval metrics:")
for k, v in preds.metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=["real", "fake"], digits=4))

print("Confusion matrix (rows=actual, cols=predicted):")
print(pd.DataFrame(
    confusion_matrix(y_true, y_pred),
    index=["actual_real", "actual_fake"],
    columns=["pred_real", "pred_fake"],
))

/opt/jupyterhub/jupyter_env/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Eval metrics:
  test_loss: 0.0631
  test_accuracy: 0.9937
  test_f1_macro: 0.9937
  test_runtime: 14.0663
  test_samples_per_second: 1022.7970
  test_steps_per_second: 8.0330

Classification report:
              precision    recall  f1-score   support

        real     0.9919    0.9953    0.9936      7006
        fake     0.9955    0.9923    0.9939      7381

    accuracy                         0.9937     14387
   macro avg     0.9937    0.9938    0.9937     14387
weighted avg     0.9938    0.9937    0.9937     14387

Confusion matrix (rows=actual, cols=predicted):
             pred_real  pred_fake
actual_real       6973         33
actual_fake         57       7324


## 12. Save the fine-tuned model

In [15]:
final_dir = OUT_DIR / "final"
trainer.save_model(str(final_dir))
tokenizer.save_pretrained(str(final_dir))
print(f"Saved fine-tuned model to: {final_dir}")
print("Load later with:")
print(f"  AutoModelForSequenceClassification.from_pretrained('{final_dir}')")
print(f"  AutoTokenizer.from_pretrained('{final_dir}')")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved fine-tuned model to: distilbert_welfake/final
Load later with:
  AutoModelForSequenceClassification.from_pretrained('distilbert_welfake/final')
  AutoTokenizer.from_pretrained('distilbert_welfake/final')


## 13. Quick inference smoke test (optional)

In [16]:
from transformers import pipeline

clf = pipeline(
    "text-classification",
    model=str(OUT_DIR / "final"),
    tokenizer=str(OUT_DIR / "final"),
    device=0 if torch.cuda.is_available() else -1,
)

examples = [
    "Scientists at a major university published peer-reviewed research showing the new vaccine produced no serious adverse effects in a 30,000-person trial.",
    "SHOCKING: Doctors HATE this one weird trick that cures cancer overnight! Big pharma doesn't want you to know!!!",
]
for e in examples:
    cleaned = clean_text(e)
    print(clf(cleaned)[0], "<<", e[:80])

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{'label': 'real', 'score': 0.9962192177772522} << Scientists at a major university published peer-reviewed research showing the ne
{'label': 'fake', 'score': 0.9999129772186279} << SHOCKING: Doctors HATE this one weird trick that cures cancer overnight! Big pha
